In [ ]:
!pip install elasticsearch googletrans==4.0.0-rc1 sentence-transformers

In [ ]:
!pip install -q elasticsearch

In [3]:
import json
import os
from elasticsearch import Elasticsearch
import numpy as np

In [4]:
import sys
sys.path.append('/kaggle/input/py-lib/AIC')

from elastic_search_processor import ElasticSearchProcessor

In [5]:
es_processor = ElasticSearchProcessor(
    "https://8641545ce73f41b3a05dbc80de48d72e.asia-northeast1.gcp.cloud.es.io:443", 
    "aHdlcWFaUUJnZjYwZ0FWNHN1Snk6THZFVl9XLS1Rb0tQcERyTkVCS08ydw==", 
    "huyrc")

success, message = es_processor.create_index_with_mapping()
print(message)

Index 'huyrc' đã tồn tại


In [6]:
es_processor.client.ping()

True

In [7]:
indexMapping = {
    "properties": {
        "video_id": {
            "type": "keyword"
        },
        "start_time": {
            "type": "float"
        },
        "end_time": {
            "type": "float"
        },
        "AudioTextVector": {
            "type": "dense_vector",
            "dims": 768,
            "index": True,
            "similarity": "l2_norm"
        }
    }
}
#response = es_processor.client.indices.delete(index='audio_features_1')
#response = es_processor.client.indices.delete(index='audio_features_2')
#response = es_processor.client.indices.delete(index='audio_features_3')
#response = es_processor.client.indices.delete(index='audio_features_4')

es_processor.client.indices.create(index="audio_features_1", mappings=indexMapping)
es_processor.client.indices.create(index="audio_features_2", mappings=indexMapping)
es_processor.client.indices.create(index="audio_features_3", mappings=indexMapping)
es_processor.client.indices.create(index="audio_features_4", mappings=indexMapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'audio_features_4'})

In [ ]:
folder_path = '/kaggle/input/full-30l-audio-features/31 audio features'
i = 0
# Loop through the folder and load each .npy file
for file_name in os.listdir(folder_path):
    if file_name.endswith('.npy'):
        # Parse the file name to extract video ID, start time, and end time
        parts = file_name.replace('.npy', '').split('_')
        video_id = parts[0] + "_" + parts[1]  # e.g., L01_V001
        start_time = float(parts[2])  # Start time from filename
        end_time = float(parts[3])  # End time from filename

        # Load the corresponding feature vector from the .npy file
        file_path = os.path.join(folder_path, file_name)
        audio_text_vector = np.load(file_path).tolist()  # Convert numpy array to list

        # Create the document to be indexed
        doc = {
            "video_id": video_id,
            "start_time": start_time,
            "end_time": end_time,
            "AudioTextVector": audio_text_vector
        }
        if i < 10000:
            # Index the document in Elasticsearch
            try:
                es_processor.client.index(index="audio_features_1", document=doc)
                print(f"Indexed {file_name} successfully")
            except Exception as e:
                print(f"Error indexing {file_name}: {e}")
        if i >= 10000 and i < 20000:
            try:
                es_processor.client.index(index="audio_features_2", document=doc)
                print(f"Indexed {file_name} successfully")
            except Exception as e:
                print(f"Error indexing {file_name}: {e}")
        if i >= 20000 and i < 30000:
            try:
                es_processor.client.index(index="audio_features_3", document=doc)
                print(f"Indexed {file_name} successfully")
            except Exception as e:
                print(f"Error indexing {file_name}: {e}")
        if i >= 30000:
            try:
                es_processor.client.index(index="audio_features_4", document=doc)
                print(f"Indexed {file_name} successfully")
            except Exception as e:
                print(f"Error indexing {file_name}: {e}")
        i += 1

In [ ]:
from googletrans import Translator
from sentence_transformers import SentenceTransformer

# Initialize translator and sentence transformer model
translator = Translator()
model = SentenceTransformer("multi-qa-mpnet-base-cos-v1")

# Function to encode and normalize the text query
def encode_text_query(query):
    translated = translator.translate(query, dest='en')  # Translate to English if necessary
    query = translated.text
    feature_vector = np.array(model.encode(query))
    feature_vector = feature_vector[np.newaxis, :]
    feature_vector = feature_vector / np.linalg.norm(feature_vector)  # Normalize the vector
    return feature_vector

# Encode the input keyword
input_keyword = "birthday party for elephants"
vector = encode_text_query(input_keyword)
doc_count_1 = es_processor.client.count(index='audio_features_1')['count']
doc_count_2 = es_processor.client.count(index='audio_features_2')['count']
doc_count_3 = es_processor.client.count(index='audio_features_3')['count']
doc_count_4 = es_processor.client.count(index='audio_features_4')['count']

# KNN search query
query_1 = {
    "field": "AudioTextVector",
    "query_vector": vector.flatten().tolist(),  # Ensure it's a list for Elasticsearch
    "k": 10,  # Return top 10 results
    "num_candidates": doc_count_1
}
query_2 = {
    "field": "AudioTextVector",
    "query_vector": vector.flatten().tolist(),  # Ensure it's a list for Elasticsearch
    "k": 10,  # Return top 10 results
    "num_candidates": doc_count_2
}
query_3 = {
    "field": "AudioTextVector",
    "query_vector": vector.flatten().tolist(),  # Ensure it's a list for Elasticsearch
    "k": 10,  # Return top 10 results
    "num_candidates": doc_count_3
}
query_4 = {
    "field": "AudioTextVector",
    "query_vector": vector.flatten().tolist(),  # Ensure it's a list for Elasticsearch
    "k": 10,  # Return top 10 results
    "num_candidates": doc_count_4
}
# Perform the search
print("First part:")
res = es_processor.client.knn_search(index="audio_features_1", knn=query_1, _source=["video_id", "start_time", "end_time"])
# Output the search results
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, Start: {hit['_source']['start_time']}, End: {hit['_source']['end_time']}")
print("Second part:")
res = es_processor.client.knn_search(index="audio_features_2", knn=query_2, _source=["video_id", "start_time", "end_time"])
# Output the search results
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, Start: {hit['_source']['start_time']}, End: {hit['_source']['end_time']}")
print("Third_part:")
res = es_processor.client.knn_search(index="audio_features_3", knn=query_3, _source=["video_id", "start_time", "end_time"])
# Output the search results
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, Start: {hit['_source']['start_time']}, End: {hit['_source']['end_time']}")
print("Fourth_part:")
res = es_processor.client.knn_search(index="audio_features_4", knn=query_4, _source=["video_id", "start_time", "end_time"])
# Output the search results
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, Start: {hit['_source']['start_time']}, End: {hit['_source']['end_time']}")